In [1]:
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype
from collections import defaultdict

In [2]:
def preprocess(df):
    return df

df1 = preprocess(pd.read_parquet('train/8.성과정보/201807_train_성과정보.parquet'))
df2 = preprocess(pd.read_parquet('train/8.성과정보/201808_train_성과정보.parquet'))
df3 = preprocess(pd.read_parquet('train/8.성과정보/201809_train_성과정보.parquet'))
df4 = preprocess(pd.read_parquet('train/8.성과정보/201810_train_성과정보.parquet'))
df5 = preprocess(pd.read_parquet('train/8.성과정보/201811_train_성과정보.parquet'))
df6 = preprocess(pd.read_parquet('train/8.성과정보/201812_train_성과정보.parquet'))

In [3]:
dfs = [df.drop(columns=['기준년월'], errors='ignore') for df in [df1, df2, df3, df4, df5, df6]]

def merge_two_avg(df_left, df_right):
    merge_keys = ['ID']
    if 'Segment' in df_left.columns and 'Segment' in df_right.columns:
        merge_keys.append('Segment')

    merged = pd.merge(df_left, df_right, on=merge_keys, how='outer', suffixes=('_left', '_right'))
    result = merged[merge_keys].copy()
    
    # 평균 계산
    for col in set(df_left.columns).union(df_right.columns):
        if col in merge_keys:
            continue
        col_left = f"{col}_left" if f"{col}_left" in merged.columns else None
        col_right = f"{col}_right" if f"{col}_right" in merged.columns else None
        
        cols_to_avg = [c for c in [col_left, col_right] if c is not None]
        result[col] = merged[cols_to_avg].mean(axis=1, skipna=True)
    
    return result

from functools import reduce
merged_df = reduce(merge_two_avg, dfs)
merged_df

,ID,증감율_이용금액_체크_분기,혜택수혜율_B0M,변동률_할부평잔,증감율_이용건수_신판_전월,잔액_신판평균한도소진율_r6m,변동률_잔액_일시불_B1M,증감율_이용금액_일시불_전월,증감율_이용건수_일시불_분기,증감율_이용금액_신판_분기,...,증감율_이용건수_카드론_분기,증감율_이용건수_신용_분기,증감율_이용건수_체크_전월,증감율_이용건수_신판_분기,잔액_신판ca최대한도소진율_r6m,잔액_신판ca최대한도소진율_r3m,증감율_이용건수_할부_전월,증감율_이용금액_일시불_분기,증감율_이용금액_CA_전월,변동률_카드론평잔
0,TRAIN_000000,-0.250000,1.444044,0.635168,-0.028249,0.097948,0.064887,0.239873,-0.292315,-0.416705,...,0.0,-0.292315,0.00000,-0.292315,0.860117,0.842530,0.000000,-0.429291,0.00000,0.999998
1,TRAIN_000001,0.000000,0.000000,0.901605,-0.137449,0.562503,-0.061162,0.168148,-0.207193,-0.293795,...,0.0,-0.207193,0.00000,-0.207193,0.708728,0.623303,0.000000,-0.293795,0.00000,0.999998
2,TRAIN_000002,0.000000,-0.482528,0.837307,-0.001158,0.216880,0.101498,-0.217274,-0.106727,-0.059676,...,0.0,-0.123001,0.00000,-0.113843,0.942631,0.883661,-0.062500,-0.051488,-0.05135,0.999998
3,TRAIN_000003,0.000000,1.789433,0.912414,0.049905,0.363748,0.120753,0.281843,0.270926,-0.202342,...,0.0,0.213005,0.00000,0.213005,1.072308,1.027524,-0.062500,-0.080496,0.00000,0.999998
4,TRAIN_000004,-0.128090,0.000000,0.999998,1.124998,0.003088,0.000000,0.999915,0.999998,0.999998,...,0.0,0.999998,-0.09004,0.999998,0.002769,0.006943,0.000000,0.999998,0.00000,0.999998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,-0.064975,NaN,0.999998,0.000000,0.000540,0.000000,0.000039,0.000000,0.000000,...,0.0,0.000000,-0.15922,0.000000,0.018348,0.027659,0.000000,0.000000,0.00000,0.999998
399996,TRAIN_399996,0.000000,2.277150,0.874998,-0.012479,0.087425,-0.115933,0.170532,-0.494337,-0.660477,...,0.0,-0.494337,0.00000,-0.494337,0.194268,0.112188,0.000000,-0.660477,0.00000,0.922704
399997,TRAIN_399997,0.000000,0.000000,0.679630,-0.009950,0.117066,0.077845,0.247223,-0.082280,0.152829,...,0.0,-0.067948,0.00000,-0.067948,0.213335,0.182148,0.062500,0.054895,0.00000,0.999998
399998,TRAIN_399998,0.000000,NaN,0.999998,0.000000,0.000000,0.000000,0.000016,0.000000,0.000000,...,0.0,0.000000,0.00000,0.000000,0.006487,0.014388,0.000000,0.000000,0.00000,0.999998


In [4]:
df_segment = pd.read_parquet('train/1.회원정보/201807_train_회원정보.parquet')[['ID', 'Segment']]

# 2. 중복 제거 (ID별 Segment가 유일하다는 전제)
df_segment = df_segment.drop_duplicates(subset='ID')

# 3. merged_df에 Segment 열 붙이기 (ID 기준)
merged_df = pd.merge(merged_df, df_segment, on='ID', how='left')
merged_df

,ID,증감율_이용금액_체크_분기,혜택수혜율_B0M,변동률_할부평잔,증감율_이용건수_신판_전월,잔액_신판평균한도소진율_r6m,변동률_잔액_일시불_B1M,증감율_이용금액_일시불_전월,증감율_이용건수_일시불_분기,증감율_이용금액_신판_분기,...,증감율_이용건수_신용_분기,증감율_이용건수_체크_전월,증감율_이용건수_신판_분기,잔액_신판ca최대한도소진율_r6m,잔액_신판ca최대한도소진율_r3m,증감율_이용건수_할부_전월,증감율_이용금액_일시불_분기,증감율_이용금액_CA_전월,변동률_카드론평잔,Segment
0,TRAIN_000000,-0.250000,1.444044,0.635168,-0.028249,0.097948,0.064887,0.239873,-0.292315,-0.416705,...,-0.292315,0.00000,-0.292315,0.860117,0.842530,0.000000,-0.429291,0.00000,0.999998,D
1,TRAIN_000001,0.000000,0.000000,0.901605,-0.137449,0.562503,-0.061162,0.168148,-0.207193,-0.293795,...,-0.207193,0.00000,-0.207193,0.708728,0.623303,0.000000,-0.293795,0.00000,0.999998,E
2,TRAIN_000002,0.000000,-0.482528,0.837307,-0.001158,0.216880,0.101498,-0.217274,-0.106727,-0.059676,...,-0.123001,0.00000,-0.113843,0.942631,0.883661,-0.062500,-0.051488,-0.05135,0.999998,C
3,TRAIN_000003,0.000000,1.789433,0.912414,0.049905,0.363748,0.120753,0.281843,0.270926,-0.202342,...,0.213005,0.00000,0.213005,1.072308,1.027524,-0.062500,-0.080496,0.00000,0.999998,D
4,TRAIN_000004,-0.128090,0.000000,0.999998,1.124998,0.003088,0.000000,0.999915,0.999998,0.999998,...,0.999998,-0.09004,0.999998,0.002769,0.006943,0.000000,0.999998,0.00000,0.999998,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,-0.064975,NaN,0.999998,0.000000,0.000540,0.000000,0.000039,0.000000,0.000000,...,0.000000,-0.15922,0.000000,0.018348,0.027659,0.000000,0.000000,0.00000,0.999998,E
399996,TRAIN_399996,0.000000,2.277150,0.874998,-0.012479,0.087425,-0.115933,0.170532,-0.494337,-0.660477,...,-0.494337,0.00000,-0.494337,0.194268,0.112188,0.000000,-0.660477,0.00000,0.922704,D
399997,TRAIN_399997,0.000000,0.000000,0.679630,-0.009950,0.117066,0.077845,0.247223,-0.082280,0.152829,...,-0.067948,0.00000,-0.067948,0.213335,0.182148,0.062500,0.054895,0.00000,0.999998,C
399998,TRAIN_399998,0.000000,NaN,0.999998,0.000000,0.000000,0.000000,0.000016,0.000000,0.000000,...,0.000000,0.00000,0.000000,0.006487,0.014388,0.000000,0.000000,0.00000,0.999998,E


In [5]:
nan_columns = merged_df.columns[merged_df.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
['혜택수혜율_B0M', '혜택수혜율_R3M']


In [8]:
ex1 = merged_df

In [9]:
cols_to_drop = ['혜택수혜율_B0M', '혜택수혜율_R3M']
ex1.drop(columns=cols_to_drop, inplace=True)

In [10]:
missing_mask = ex1.isna() | (ex1 == -1)
missing_ratio = missing_mask.mean()

high_na = missing_ratio[missing_ratio > 0.2].index.tolist()

high_const_cols = []
threshold_const = 0.8

for col in ex1.columns:
    top_ratio = ex1[col].value_counts(normalize=True, dropna=False).values[0]
    if top_ratio > threshold_const:
        high_const_cols.append(col)

to_drop = list(set(high_na + high_const_cols))

if 'Segment' in to_drop:
    to_drop.remove('Segment')

print("삭제 대상 컬럼 (결측>20% 또는 동일값>80%):", to_drop)

ex1.drop(columns=to_drop, inplace=True)

삭제 대상 컬럼 (결측>20% 또는 동일값>80%): ['증감율_이용건수_체크_전월', '증감율_이용금액_체크_분기', '증감율_이용금액_카드론_전월', '증감율_이용건수_카드론_전월', '변동률_잔액_CA_B1M', '증감율_이용건수_할부_전월', '증감율_이용건수_CA_분기', '변동률_카드론평잔', '증감율_이용금액_CA_분기', '증감율_이용건수_체크_분기', '증감율_이용건수_CA_전월', '증감율_이용금액_CA_전월', '증감율_이용금액_체크_전월', '증감율_이용건수_카드론_분기', '증감율_이용금액_카드론_분기', '변동률_CA평잔', '변동률_RVCA평잔']


In [11]:
num_df = ex1.select_dtypes(include=[np.number]).dropna()

corr = num_df.corr().abs()

high_corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)
high_corr_pairs.columns = ['Feature_1', 'Feature_2', 'Correlation']
high_corr_pairs = high_corr_pairs[high_corr_pairs['Correlation'] > 0.8]

if not np.issubdtype(ex1['Segment'].dtype, np.number):
    segment_map = {label: idx for idx, label in enumerate(sorted(ex1['Segment'].unique()))}
    ex1['Segment_encoded'] = ex1['Segment'].map(segment_map)
else:
    ex1['Segment_encoded'] = ex1['Segment']

segment_corr = ex1[num_df.columns].corrwith(ex1['Segment_encoded']).abs()

high_corr_pairs['Corr_with_Segment_1'] = high_corr_pairs['Feature_1'].map(segment_corr)
high_corr_pairs['Corr_with_Segment_2'] = high_corr_pairs['Feature_2'].map(segment_corr)

high_corr_pairs = high_corr_pairs.sort_values(by='Correlation', ascending=False).reset_index(drop=True)

print(f"▶ 상관계수 0.7 초과 변수쌍 수: {len(high_corr_pairs)}")
display(high_corr_pairs)


▶ 상관계수 0.7 초과 변수쌍 수: 50


,Feature_1,Feature_2,Correlation,Corr_with_Segment_1,Corr_with_Segment_2
0,잔액_신판평균한도소진율_r3m,잔액_신판최대한도소진율_r3m,0.996141,0.190783,0.195629
1,잔액_신판ca평균한도소진율_r3m,잔액_신판ca최대한도소진율_r3m,0.995650,0.261063,0.266437
2,잔액_신판ca평균한도소진율_r6m,잔액_신판ca최대한도소진율_r6m,0.989298,0.254131,0.269636
3,잔액_신판최대한도소진율_r3m,잔액_신판최대한도소진율_r6m,0.989179,0.195629,0.200991
4,잔액_신판ca최대한도소진율_r6m,잔액_신판ca최대한도소진율_r3m,0.988714,0.269636,0.266437
5,잔액_신판평균한도소진율_r6m,잔액_신판최대한도소진율_r6m,0.988377,0.182581,0.200991
6,잔액_신판평균한도소진율_r6m,잔액_신판평균한도소진율_r3m,0.988350,0.182581,0.190783
7,증감율_이용건수_신판_전월,증감율_이용건수_신용_전월,0.986098,0.003705,0.002007
8,잔액_신판ca평균한도소진율_r6m,잔액_신판ca평균한도소진율_r3m,0.985452,0.254131,0.261063
9,잔액_신판평균한도소진율_r6m,잔액_신판최대한도소진율_r3m,0.985038,0.182581,0.195629


In [12]:
to_drop = []

for _, row in high_corr_pairs.iterrows():
    f1, f2 = row['Feature_1'], row['Feature_2']
    c1, c2 = row['Corr_with_Segment_1'], row['Corr_with_Segment_2']
    
    if pd.isna(c1) or pd.isna(c2):
        continue
    
    if c1 < c2:
        to_drop.append(f1)
    else:
        to_drop.append(f2)

to_drop = list(set(to_drop))

# 결과 출력
print(f"▶ 제거 대상 피처 수: {len(to_drop)}")
print("제거할 피처 목록:")
print(to_drop)

▶ 제거 대상 피처 수: 17
제거할 피처 목록:
['증감율_이용금액_할부_분기', '잔액_신판최대한도소진율_r6m', '증감율_이용건수_신판_분기', '증감율_이용건수_신판_전월', '잔액_신판ca최대한도소진율_r3m', '잔액_신판평균한도소진율_r6m', '잔액_신판ca평균한도소진율_r3m', '증감율_이용금액_신용_전월', '증감율_이용건수_신용_전월', '증감율_이용금액_일시불_전월', '증감율_이용금액_일시불_분기', '잔액_신판평균한도소진율_r3m', '증감율_이용금액_신판_분기', '증감율_이용건수_일시불_분기', '잔액_신판최대한도소진율_r3m', '증감율_이용금액_신용_분기', '잔액_신판ca평균한도소진율_r6m']


In [13]:
cols_to_drop = ['증감율_이용금액_할부_분기', '잔액_신판최대한도소진율_r6m', '증감율_이용건수_신판_분기', '증감율_이용건수_신판_전월', '잔액_신판ca최대한도소진율_r3m', '잔액_신판평균한도소진율_r6m', '잔액_신판ca평균한도소진율_r3m', '증감율_이용금액_신용_전월', '증감율_이용건수_신용_전월', '증감율_이용금액_일시불_전월', '증감율_이용금액_일시불_분기', '잔액_신판평균한도소진율_r3m', '증감율_이용금액_신판_분기', '증감율_이용건수_일시불_분기', '잔액_신판최대한도소진율_r3m', '증감율_이용금액_신용_분기', '잔액_신판ca평균한도소진율_r6m']
ex1.drop(columns=cols_to_drop, inplace=True)

In [14]:
cols_to_drop = ['Segment_encoded']
ex1.drop(columns=cols_to_drop, inplace=True)
cols = ex1.columns.tolist()
cols

['ID',
 '변동률_할부평잔',
 '변동률_잔액_일시불_B1M',
 '변동률_잔액_B1M',
 '증감율_이용금액_신판_전월',
 '변동률_RV일시불평잔',
 '증감율_이용건수_할부_분기',
 '변동률_일시불평잔',
 '증감율_이용금액_할부_전월',
 '증감율_이용건수_일시불_전월',
 '증감율_이용건수_신용_분기',
 '잔액_신판ca최대한도소진율_r6m',
 'Segment']

In [15]:
ex1.to_parquet('성과_전처리_Segment.parquet', index=False)

In [16]:
def preprocess(df):
    return df

ddf1 = preprocess(pd.read_parquet('train/8.성과정보/201807_train_성과정보.parquet'))
ddf2 = preprocess(pd.read_parquet('train/8.성과정보/201808_train_성과정보.parquet'))
ddf3 = preprocess(pd.read_parquet('train/8.성과정보/201809_train_성과정보.parquet'))
ddf4 = preprocess(pd.read_parquet('train/8.성과정보/201810_train_성과정보.parquet'))
ddf5 = preprocess(pd.read_parquet('train/8.성과정보/201811_train_성과정보.parquet'))
ddf6 = preprocess(pd.read_parquet('train/8.성과정보/201812_train_성과정보.parquet'))

In [17]:

dfs = [ddf1, ddf2, ddf3, ddf4, ddf5, ddf6]
for i in range(len(dfs)):
    if '기준년월' in dfs[i].columns:
        dfs[i] = dfs[i].drop(columns=['기준년월'])

# ID 기준으로 병합 후 평균
from functools import reduce

merged_df = reduce(
    lambda left, right: pd.merge(left, right, on='ID', how='outer', suffixes=('', '_dup')),
    dfs
)

# 같은 이름의 열 평균 구하기
from collections import defaultdict
import pandas as pd

result = pd.DataFrame()
result['ID'] = merged_df['ID']

# 열 이름 모아 평균 구하기
col_dict = defaultdict(list)
for col in merged_df.columns:
    if col != 'ID':
        base_col = col.split('_dup')[0]
        col_dict[base_col].append(col)

for base_col, cols in col_dict.items():
    result[base_col] = merged_df[cols].mean(axis=1, skipna=True)

# 결과 확인
print(result.head())

             ID  증감율_이용건수_신용_전월  증감율_이용건수_신판_전월  증감율_이용건수_일시불_전월  \
0  TRAIN_000000       -0.025486       -0.025486        -0.031738   
1  TRAIN_000001       -0.133216       -0.133216        -0.133216   
2  TRAIN_000002        0.014527        0.014527         0.019548   
3  TRAIN_000003        0.081703        0.081703         0.082405   
4  TRAIN_000004        0.769229        0.769229         0.769229   

   증감율_이용건수_할부_전월  증감율_이용건수_CA_전월  증감율_이용건수_체크_전월  증감율_이용건수_카드론_전월  \
0       -0.307692             0.0        0.000000              0.0   
1        0.000000             0.0        0.000000              0.0   
2       -0.076923             0.0        0.000000              0.0   
3       -0.076923             0.0        0.000000              0.0   
4        0.000000             0.0       -0.128848              0.0   

   증감율_이용금액_신용_전월  증감율_이용금액_신판_전월  ...  변동률_RV일시불평잔  변동률_할부평잔  변동률_CA평잔  \
0        0.236688        0.236688  ...     0.999998  0.698589  1.000817   
1        0.158473   

In [18]:
cols = ['ID',
 '변동률_할부평잔',
 '변동률_잔액_일시불_B1M',
 '변동률_잔액_B1M',
 '증감율_이용금액_신판_전월',
 '변동률_RV일시불평잔',
 '증감율_이용건수_할부_분기',
 '변동률_일시불평잔',
 '증감율_이용금액_할부_전월',
 '증감율_이용건수_일시불_전월',
 '증감율_이용건수_신용_분기',
 '잔액_신판ca최대한도소진율_r6m']

result = result[cols]
result

,ID,변동률_할부평잔,변동률_잔액_일시불_B1M,변동률_잔액_B1M,증감율_이용금액_신판_전월,변동률_RV일시불평잔,증감율_이용건수_할부_분기,변동률_일시불평잔,증감율_이용금액_할부_전월,증감율_이용건수_일시불_전월,증감율_이용건수_신용_분기,잔액_신판ca최대한도소진율_r6m
0,TRAIN_000000,0.698589,0.034151,-0.062315,0.236688,0.999998,0.000000,0.840517,-0.307692,-0.031738,-0.355093,0.849401
1,TRAIN_000001,0.902540,-0.064556,-0.084391,0.158473,1.004224,0.000000,1.036596,0.000000,-0.133216,-0.050501,0.746933
2,TRAIN_000002,1.185772,0.127707,0.022114,0.288537,0.987809,-0.692306,1.034022,0.346299,0.019548,0.062231,0.957420
3,TRAIN_000003,0.995150,0.144485,0.077657,0.291796,0.999998,-0.319263,0.954274,-0.182313,0.082405,0.170893,1.072257
4,TRAIN_000004,0.999998,0.000000,0.000000,0.384673,0.999998,0.000000,0.384615,0.000000,0.769229,0.307692,0.001065
...,...,...,...,...,...,...,...,...,...,...,...,...
399995,TRAIN_399995,0.999998,0.000000,0.000000,0.000041,0.999998,0.000000,0.999998,0.000000,0.000000,0.000000,0.013288
399996,TRAIN_399996,0.576922,-0.086141,-0.119043,0.161035,0.999998,0.000000,0.820432,0.000000,-0.042182,-0.307138,0.222879
399997,TRAIN_399997,0.460850,0.060781,0.057399,0.274901,0.999998,1.076921,1.068625,-0.307692,0.000692,0.083068,0.230979
399998,TRAIN_399998,0.999998,0.000000,0.000000,0.000027,0.999998,0.000000,0.999998,0.000000,0.000000,0.000000,0.002620


In [19]:
result.to_parquet('성과_전처리_test.parquet', index=False)

In [20]:
nan_columns = result.columns[result.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
[]
